In [1]:
import numpy as np
import xarray as xr
import owncloud
from pathlib import Path
import pandas as pd

import torch
import torch.nn as nn

In [2]:
Path('data').mkdir(exist_ok=True, parents=True)

owncloud.Client.from_public_link('https://uni-bonn.sciebo.de/s/3Uf2gScrvuTPQhB').get_file('/', f'data/steinmetz_2017-01-08_Muller.nc')


True

In [2]:
dset = xr.load_dataset('data/steinmetz_2017-01-08_Muller.nc')
dset

<xarray.Dataset> Size: 124MB
Dimensions:             (trial: 261, time: 250, cell: 1268, sample: 82,
                         waveform_component: 3, probe: 384, brain_area_lfp: 5,
                         spike_id: 1836009)
Coordinates:
  * trial               (trial) int32 1kB 1 2 3 4 5 6 ... 257 258 259 260 261
  * time                (time) float64 2kB 0.01 0.02 0.03 0.04 ... 2.48 2.49 2.5
  * cell                (cell) int32 5kB 1 2 3 4 5 ... 1264 1265 1266 1267 1268
  * waveform_component  (waveform_component) int32 12B 1 2 3
  * probe               (probe) int32 2kB 1 2 3 4 5 6 ... 380 381 382 383 384
  * brain_area_lfp      (brain_area_lfp) <U5 100B 'CA1' 'DG' 'LP' 'PO' 'VISam'
  * spike_id            (spike_id) int32 7MB 1 2 3 4 ... 1836007 1836008 1836009
Dimensions without coordinates: sample
Data variables: (12/31)
    contrast_left       (trial) int8 261B 50 0 100 0 50 0 ... 100 0 100 0 100 0
    contrast_right      (trial) int8 261B 0 50 25 100 50 50 ... 100 50 100 25 25
    gocue               (trial) float64 2kB 0.9828 0.902 1.114 ... nan nan nan
    stim_onset          (trial) float64 2kB 0.5 0.5 0.5 0.5 ... 0.5 0.5 0.5 0.5
    feedback_type       (trial) float64 2kB 1.0 1.0 1.0 1.0 ... nan nan nan nan
    feedback_time       (trial) float64 2kB 1.272 1.104 1.402 ... nan nan nan
    ...                  ...
    waveform_w          (cell, sample, waveform_component) float32 1MB 0.0 .....
    waveform_u          (cell, waveform_component, probe) float32 6MB 0.0 ......
    lfp                 (brain_area_lfp, trial, time) float64 3MB -27.6 ... 0...
    spike_time          (spike_id) float32 7MB 2.363 2.385 ... 1.651 0.5142
    spike_cell          (spike_id) uint32 7MB 1 1 1 1 1 ... 1268 1268 1268 1268
    spike_trial         (spike_id) uint32 7MB 1 1 2 2 2 ... 205 205 205 213 252
Attributes:
    session_date:  2017-01-08
    mouse:         Muller
    stim_onset:    0.5
    bin_size:      0.01

Classification

In [3]:
spike_cols = ['spike_time', 'spike_cell', 'spike_trial']
contrast_cols = ['contrast_left', 'contrast_right']
df_spikes = dset[spike_cols].to_dataframe().reset_index()
df_contrast = dset[contrast_cols].to_dataframe().reset_index()

trial_ids = np.sort(df_spikes['spike_trial'].unique())


### Train a Classifier to Decode Stimulus Information from Spike Data

Make features out of spike data and labels for trials out of stimulus contrast

In [4]:

# classify trials based on which side had higher contrast
trial_features = []
trial_labels = []
for trial_id in trial_ids:
    trial_spikes = df_spikes[df_spikes['spike_trial'] == trial_id]
    
    # Count spikes per cell, convert to firing rate
    spikes_per_cell = trial_spikes.groupby('spike_cell').size()
    feature_vector = np.zeros(1268)
    for cell_id, count in spikes_per_cell.items():
        feature_vector[cell_id - 1] = count  # cell_id starts at 1
    
    trial_features.append(feature_vector)
    
    # Label: which side had higher contrast
    left = dset['contrast_left'].values[trial_id - 1]
    right = dset['contrast_right'].values[trial_id - 1]

    if left > right:
        trial_labels.append(0)
    elif right > left:
        trial_labels.append(1)
    else:
        trial_labels.append(2)

Make features and labels tensors

In [5]:
features = torch.tensor(np.array(trial_features), dtype=torch.float32)
labels = torch.tensor(np.array(trial_labels), dtype=torch.long)

Create model

In [6]:
torch.manual_seed(2025)
model = nn.Sequential(
    nn.Linear(1268, 64),
    nn.ReLU(),
    nn.Linear(64, 3)
)

Train

In [7]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(100):
    optimizer.zero_grad()
    labels_pred = model(features)
    loss = loss_fn(labels_pred, labels)
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        acc = (torch.argmax(labels_pred, dim=1) == labels).float().mean()
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Accuracy: {acc.item():.4f}")

Epoch 0, Loss: 1.8470, Accuracy: 0.3908
Epoch 20, Loss: 0.8655, Accuracy: 0.6169
Epoch 40, Loss: 0.5974, Accuracy: 0.8008
Epoch 60, Loss: 0.4064, Accuracy: 0.8774
Epoch 80, Loss: 0.2685, Accuracy: 0.9349


Split into train and test data

In [8]:
from sklearn.model_selection import train_test_split

features_train, features_test, labels_train, labels_test = train_test_split(
    features, labels, train_size=0.8, test_size=0.2, random_state=2025
)

Create model

In [9]:
torch.manual_seed(2025)
model = nn.Sequential(
    nn.Linear(1268, 64),
    nn.ReLU(),
    nn.Linear(64, 3)
)

Train

In [10]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(400):
    optimizer.zero_grad()
    labels_pred = model(features_train)
    loss = loss_fn(labels_pred, labels_train)
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        acc = (torch.argmax(labels_pred, dim=1) == labels_train).float().mean()
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Accuracy: {acc.item():.4f}")

Epoch 0, Loss: 1.7474, Accuracy: 0.4038
Epoch 20, Loss: 0.7348, Accuracy: 0.7260
Epoch 40, Loss: 0.3916, Accuracy: 0.8846
Epoch 60, Loss: 0.1921, Accuracy: 0.9615
Epoch 80, Loss: 0.0949, Accuracy: 1.0000
Epoch 100, Loss: 0.0525, Accuracy: 1.0000
Epoch 120, Loss: 0.0329, Accuracy: 1.0000
Epoch 140, Loss: 0.0228, Accuracy: 1.0000
Epoch 160, Loss: 0.0168, Accuracy: 1.0000
Epoch 180, Loss: 0.0130, Accuracy: 1.0000
Epoch 200, Loss: 0.0104, Accuracy: 1.0000
Epoch 220, Loss: 0.0086, Accuracy: 1.0000
Epoch 240, Loss: 0.0072, Accuracy: 1.0000
Epoch 260, Loss: 0.0061, Accuracy: 1.0000
Epoch 280, Loss: 0.0053, Accuracy: 1.0000
Epoch 300, Loss: 0.0046, Accuracy: 1.0000
Epoch 320, Loss: 0.0040, Accuracy: 1.0000
Epoch 340, Loss: 0.0036, Accuracy: 1.0000
Epoch 360, Loss: 0.0032, Accuracy: 1.0000
Epoch 380, Loss: 0.0029, Accuracy: 1.0000


Test

In [11]:
with torch.no_grad():
    labels_pred = model(features_test)
    acc = (torch.argmax(labels_pred, dim=1) == labels_test).float().mean()
    print(f"Loss: {loss.item():.4f}, Accuracy: {acc.item():.4f}")

Loss: 0.0026, Accuracy: 0.6226


Create more advanced model

In [12]:
torch.manual_seed(2025)
model = nn.Sequential(
    nn.Linear(1268, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 32),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(32, 3)
)

Train

In [13]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(200):
    optimizer.zero_grad()
    labels_pred = model(features_train)
    loss = loss_fn(labels_pred, labels_train)
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        accuracy = (torch.argmax(labels_pred, dim=1) == labels_train).float().mean()
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Accuracy: {accuracy.item():.4f}")

Epoch 0, Loss: 1.3173, Accuracy: 0.4087
Epoch 20, Loss: 1.0430, Accuracy: 0.4760
Epoch 40, Loss: 0.9960, Accuracy: 0.4615
Epoch 60, Loss: 0.9594, Accuracy: 0.4856
Epoch 80, Loss: 0.9085, Accuracy: 0.4808
Epoch 100, Loss: 0.8693, Accuracy: 0.5000
Epoch 120, Loss: 0.7345, Accuracy: 0.6346
Epoch 140, Loss: 0.3617, Accuracy: 0.8606
Epoch 160, Loss: 0.1694, Accuracy: 0.9375
Epoch 180, Loss: 0.0921, Accuracy: 0.9615


Test

In [14]:
with torch.no_grad():
    labels_pred = model(features_test)
    acc = (torch.argmax(labels_pred, dim=1) == labels_test).float().mean()
    print(f"Test Accuracy: {acc.item():.4f}")

Test Accuracy: 0.5660


#### Notes:

(Different variants of) more advanced model didn't help with the test accuracy. The dataset is probably too small and the model just memorizes the training data.

## Section 2: Classify Neural Spike Data

In Section 1, we saw how neural networks can capture non-linear decision boundaries using synthetic data. Now let's apply what we've learned to real neuroscience data. We'll train a classifier to decode which visual stimulus was presented to a mouse based on the activity of neurons recorded during the experiment. The task is to predict whether the left stimulus had higher contrast, the right stimulus had higher contrast, or both had equal contrast—a three-class classification problem.

| Code | Description |
| --- | --- |
| `nn.Linear(n_in, n_out)` | A layer that takes `n_in` input features and outputs `n_out` features. |
| `nn.ReLU()` | Rectified linear unit activation function. |
| `nn.Sequential(...)` | Container that stacks layers in sequential order. |
| `nn.CrossEntropyLoss()` | Loss function for multiclass classification. |
| `torch.optim.Adam(model.parameters(), lr=0.001)` | Adam optimizer with specified learning rate. |
| `optimizer.zero_grad()` | Resets gradients before computing new ones. |
| `loss.backward()` | Computes gradients via backpropagation. |
| `optimizer.step()` | Updates model parameters using computed gradients. |
| `torch.argmax(output, dim=1)` | Returns index of maximum value along dimension 1 (predicted class). |
| `(predictions == labels).float().mean()` | Calculates accuracy as proportion of correct predictions. |

#### **Exercises**

**Demo**: Examine the shape of the features and labels to understand the data. How many trials are there? How many neurons (features) per trial? How many classes?

In [ ]:
print(f"Features shape: {features.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Unique labels: {torch.unique(labels)}")

**Exercise**: Create a neural network model with one hidden layer. The input size should match the number of neurons (1268), the hidden layer should have 64 neurons with ReLU activation, and the output should have 3 neurons (one per class). Assign the model to a variable named `model`.

In [ ]:
torch.manual_seed(2025)
model = nn.Sequential(
    nn.Linear(1268, 64),
    nn.ReLU(),
    nn.Linear(64, 3)
)
model

**Demo**: Set up the optimizer and loss function for training. For multiclass classification, we use `nn.CrossEntropyLoss()`. The Adam optimizer is a good default choice.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

**Exercise**: Complete the training loop below by filling in the blanks. The loop should: (1) reset gradients, (2) get predictions from the model, (3) calculate the loss, (4) do backpropagation, and (5) update the parameters.

In [ ]:
for epoch in range(100):
    ___  # reset gradients
    labels_pred = ___  # get predictions
    loss = ___  # calculate loss
    ___  # backpropagation
    ___  # update parameters
    
    if epoch % 20 == 0:
        acc = (torch.argmax(labels_pred, dim=1) == labels).float().mean()
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Accuracy: {acc.item():.4f}")

In [ ]:
# solution
torch.manual_seed(2025)
model = nn.Sequential(
    nn.Linear(1268, 64),
    nn.ReLU(),
    nn.Linear(64, 3)
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(100):
    optimizer.zero_grad()
    labels_pred = model(features)
    loss = loss_fn(labels_pred, labels)
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        acc = (torch.argmax(labels_pred, dim=1) == labels).float().mean()
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Accuracy: {acc.item():.4f}")

**Exercise**: The model above reaches high accuracy. Try changing the hidden layer size from 64 to 16. Does the model still learn? What about with 256 neurons?

In [ ]:
torch.manual_seed(2025)
model = nn.Sequential(
    nn.Linear(1268, ___),  # try 16 or 256
    nn.ReLU(),
    nn.Linear(___, 3)
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(100):
    optimizer.zero_grad()
    labels_pred = model(features)
    loss = loss_fn(labels_pred, labels)
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        acc = (torch.argmax(labels_pred, dim=1) == labels).float().mean()
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Accuracy: {acc.item():.4f}")

**Exercise**: Try changing the learning rate from 0.001 to 0.01 or 0.0001. How does this affect how quickly the model learns?

In [ ]:
torch.manual_seed(2025)
model = nn.Sequential(
    nn.Linear(1268, 64),
    nn.ReLU(),
    nn.Linear(64, 3)
)

optimizer = torch.optim.Adam(model.parameters(), lr=___)  # try 0.01 or 0.0001
loss_fn = nn.CrossEntropyLoss()

for epoch in range(100):
    optimizer.zero_grad()
    labels_pred = model(features)
    loss = loss_fn(labels_pred, labels)
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        acc = (torch.argmax(labels_pred, dim=1) == labels).float().mean()
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Accuracy: {acc.item():.4f}")

The model achieves over 90% accuracy on the training data. But does this mean it has truly learned to decode neural activity, or has it simply memorized the training examples? To answer this question, we need to evaluate the model on data it has never seen before. We'll learn how to do this properly on Day 2 when we cover train/test splitting.